# Customer Churn PredictionPredicting whether a telecom customer will churn (cancel their subscription) based on account and usage attributes.**Goal:** Build and compare classification models to identify customers at risk of churning, so the business can target them with retention offers.**Tech stack:** Python, pandas, NumPy, scikit-learn, matplotlib, seaborn

## 1. Load the data

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsdf = pd.read_csv("customer_churn.csv")df.head()

In [ ]:
df.info()df.describe()

## 2. Exploratory Data Analysis

In [ ]:
print(df['Churn'].value_counts(normalize=True))plt.figure(figsize=(5,4))sns.countplot(data=df, x='Churn')plt.title('Churn Distribution')plt.show()

In [ ]:
plt.figure(figsize=(6,4))sns.boxplot(data=df, x='Churn', y='MonthlyCharges')plt.title('Monthly Charges vs Churn')plt.show()

In [ ]:
plt.figure(figsize=(6,4))sns.boxplot(data=df, x='Churn', y='Tenure')plt.title('Tenure (months) vs Churn')plt.show()

In [ ]:
plt.figure(figsize=(6,4))sns.countplot(data=df, x='ContractType', hue='Churn')plt.title('Contract Type vs Churn')plt.show()

In [ ]:
plt.figure(figsize=(8,6))numeric_cols = df.select_dtypes(include=np.number).columnssns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')plt.title('Correlation Heatmap')plt.show()

**Observations:**- Customers on month-to-month contracts churn far more than those on annual/two-year contracts.- Shorter tenure and higher monthly charges are associated with higher churn.- These patterns match real-world telecom churn behavior, which makes them useful signals for a classifier.

## 3. Preprocessing

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScaler, LabelEncoderdata = df.copy()# Encode simple Yes/No columnsfor col in ['Partner', 'PaperlessBilling']:    data[col] = data[col].map({'Yes': 1, 'No': 0})# Encode multi-category columnscat_cols = ['ContractType', 'InternetService', 'PaymentMethod']data = pd.get_dummies(data, columns=cat_cols, drop_first=True)le = LabelEncoder()data['Churn'] = le.fit_transform(data['Churn'])  # Yes/No -> 1/0X = data.drop(columns=['CustomerID', 'Churn'])y = data['Churn']X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)scaler = StandardScaler()num_cols = ['Tenure', 'MonthlyCharges', 'TotalCharges']X_train[num_cols] = scaler.fit_transform(X_train[num_cols])X_test[num_cols] = scaler.transform(X_test[num_cols])X_train.shape, X_test.shape

## 4. Model training: Logistic Regression vs Random Forest

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import (    accuracy_score, precision_score, recall_score, f1_score,    confusion_matrix, classification_report, roc_auc_score, roc_curve)models = {    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)}results = []fitted = {}for name, model in models.items():    model.fit(X_train, y_train)    preds = model.predict(X_test)    probs = model.predict_proba(X_test)[:, 1]    results.append({        "Model": name,        "Accuracy": accuracy_score(y_test, preds),        "Precision": precision_score(y_test, preds),        "Recall": recall_score(y_test, preds),        "F1-score": f1_score(y_test, preds),        "ROC-AUC": roc_auc_score(y_test, probs)    })    fitted[name] = (model, preds, probs)results_df = pd.DataFrame(results).set_index("Model").round(3)results_df

## 5. Model evaluation

In [ ]:
best_name = results_df["F1-score"].idxmax()best_model, best_preds, best_probs = fitted[best_name]print(f"Best model by F1-score: {best_name}")print(classification_report(y_test, best_preds, target_names=['No Churn', 'Churn']))

In [ ]:
cm = confusion_matrix(y_test, best_preds)plt.figure(figsize=(5,4))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])plt.xlabel('Predicted')plt.ylabel('Actual')plt.title(f'Confusion Matrix - {best_name}')plt.show()

In [ ]:
plt.figure(figsize=(6,5))for name, (model, preds, probs) in fitted.items():    fpr, tpr, _ = roc_curve(y_test, probs)    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, probs):.2f})")plt.plot([0, 1], [0, 1], linestyle='--', color='gray')plt.xlabel('False Positive Rate')plt.ylabel('True Positive Rate')plt.title('ROC Curve Comparison')plt.legend()plt.show()

## 6. Feature importance (Random Forest)

In [ ]:
rf_model = fitted["Random Forest"][0]importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)plt.figure(figsize=(7,5))importances.head(10).plot(kind='barh')plt.gca().invert_yaxis()plt.title('Top 10 Feature Importances')plt.xlabel('Importance')plt.show()

## 7. Conclusion- Compared Logistic Regression and Random Forest for predicting customer churn.- The best-performing model reached the accuracy / F1-score / ROC-AUC shown in the results table above (fill in your actual run's numbers here once you execute the notebook, e.g. *"Random Forest achieved 84% accuracy and an F1-score of 0.79"*).- Contract type, tenure, and monthly charges were the strongest predictors of churn — this suggests retention efforts should prioritize month-to-month customers with high charges.**Next steps:** try XGBoost/LightGBM, handle class imbalance with SMOTE, and tune hyperparameters with GridSearchCV.